# AI Procurement Counts: Exploratory Forecasting

This notebook analyzes observed AI-related procurement records in the World Bank AI Procurement Contracts workbook. Counts are **records present in this workbook**, not verified totals for all contracts from each source. Source coverage may vary over time. The workbook's classification provenance is still unverified (`model` is `deepseek` in the official file although the catalog description says GPT-5).

The notebook forecasts Open Contract and World Bank source counts separately. TED is excluded because only one year is present. It uses an expanding-window, one-year-ahead rolling backtest and compares FLAML with naïve baselines. All results are exploratory, unregistered, and not for procurement decisions.

> The workbook's latest notice year is 2025. A one-step estimate beyond 2025 is **not** a validated full-year 2026 forecast. It is an estimate based on available records; actual 2026 outcomes are not available in this workbook.

In [ ]:
from pathlib import Path
import json
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "worldbank_demo_backend").is_dir():
    raise FileNotFoundError(
        "Open this notebook from the project root (worldbank_demo_backend)."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

WORKBOOK = PROJECT_ROOT / "examples" / "AI_Procurement_Contracts.xlsx"
if not WORKBOOK.is_file():
    raise FileNotFoundError(f"Expected dataset not found: {WORKBOOK}")

os.environ.setdefault(
    "MLFLOW_TRACKING_URI",
    f"sqlite:///{(PROJECT_ROOT / 'data' / 'processed' / 'mlflow.db').as_posix()}",
)
os.environ.setdefault(
    "MLFLOW_AI_PROCUREMENT_EXPERIMENT",
    "ai-procurement-exploratory-forecast",
)

print(f"Project: {PROJECT_ROOT}")
print(f"Workbook: {WORKBOOK}")
print(f"MLflow tracking URI: {os.environ['MLFLOW_TRACKING_URI']}")

## Validate, aggregate, and check source coverage

Missing notice dates are excluded, not imputed. Missing categories remain labeled `Unknown`. Missing source-years are treated as unknown coverage, never as zero contracts.

In [ ]:
from worldbank_demo_backend.ai_procurement import (
    aggregate_annual_counts,
    aggregate_source_year_counts,
    backtest_source_baselines,
    load_workbook_rows,
    prepare_contract_rows,
    source_coverage_summary,
)

raw_rows = load_workbook_rows(WORKBOOK)
dated_rows, validation = prepare_contract_rows(raw_rows)
annual_counts = aggregate_annual_counts(dated_rows)
source_year_counts = aggregate_source_year_counts(dated_rows)
coverage = source_coverage_summary(source_year_counts)
baseline_report = backtest_source_baselines(source_year_counts)

print(json.dumps(validation, indent=2, ensure_ascii=False))
print("\nObserved source coverage:")
print(json.dumps(coverage, indent=2, ensure_ascii=False))
print("\nBaseline rolling-origin comparison:")
for result in baseline_report["results"]:
    print(json.dumps(result, indent=2, ensure_ascii=False))

## Train exploratory FLAML forecasts and log with MLflow

Each source is evaluated separately over up to five expanding-window one-year-ahead origins, with at least eight years of training history per origin. The final model is then fit to that source's full observed series to produce a one-step estimate. FLAML models and metrics are logged to MLflow; the code does not register models. If FLAML/MLflow dependencies are missing, install `requirements.txt` in the notebook's Python 3.11/3.12 environment and rerun the kernel.

In [ ]:
from worldbank_demo_backend.ai_procurement import train_ai_procurement_forecasts

TIME_BUDGET_SECONDS = 10
forecast_report = train_ai_procurement_forecasts(
    source_year_counts,
    baseline_report,
    experiment_name=os.environ["MLFLOW_AI_PROCUREMENT_EXPERIMENT"],
    time_budget_seconds=TIME_BUDGET_SECONDS,
)

output_dir = PROJECT_ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
forecast_path = output_dir / "ai_procurement_flaml_forecasts.json"
forecast_path.write_text(
    json.dumps(forecast_report, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(f"MLflow run ID: {forecast_report['mlflow_run_id']}")
print(f"Forecast report: {forecast_path}")
for source, result in forecast_report["sources"].items():
    print(f"\n{source}")
    print("rolling-origin metrics:", json.dumps(result["rolling_origin_metrics"], indent=2))
    print(
        "one-step estimate:",
        json.dumps(result["next_unobserved_year_estimate"], indent=2),
    )

## Interpretation and release checklist

- The rolling-origin metrics summarize a small number of historical one-year-ahead tests; they do not prove stable future accuracy.
- Compare FLAML MAE/WAPE with both baseline methods for each source. A lower error in this small backtest is not sufficient for production approval.
- The reported target year is one year after the latest year in the workbook. If the workbook ends in 2025, the 2026 figure is an unvalidated estimate, not actual 2026 performance or a validated full-year forecast.
- These are observed dataset counts, not complete source-wide contract totals. Investigate source-specific coverage changes before interpreting trends.
- Provenance remains unverified. MLflow tags mark the run `provenance_unverified` and `exploratory_only_not_for_decisions`; models are not registered.

Before any operational use: obtain documentation clarifying the `model` field, confirm source coverage and date completeness, evaluate additional vintages as actual data arrives, compare against baselines, review subgroup/source behavior, and obtain human approval. Until then, do not use the estimates for procurement decisions.